# Multi-Modal Neural Network for Construction Cost Prediction

This notebook builds a model combining:
- Tabular data (MLP)
- Sentinel-2 imagery (CNN)
- VIIRS imagery (CNN)

We fuse all modalities into a final regression model.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
import tifffile as tiff

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Dataset handling
Loading and preparing dataset for the model

In [2]:
DataPath = Path("..") / "Processed data"
ImgPath = Path("..") / "Training data\\train_composite"

train_df = pd.read_csv(DataPath / "processed_data.csv")
philippines_df = pd.read_csv(DataPath / "processed_philippines.csv")
japan_df = pd.read_csv(DataPath / "processed_japan.csv")

print(f"Train shape: {train_df.shape}")
print(f"Japan shape: {japan_df.shape}")
print(f"Philippines shape: {philippines_df.shape}")
train_df.head()

Train shape: (1024, 21)
Japan shape: (567, 18)
Philippines shape: (457, 20)


,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,landlocked,region_economic_classification,access_to_airport,access_to_port,...,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd
0,0,3,0,0.0,0.002133,0.059541,0,1,0,1,...,0,0.496774,2,1,3,0,0,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420
1,1,2,1,1.0,0.919424,0.973574,1,3,0,0,...,1,0.238710,2,1,1,0,3,sentinel_2_29000_nara_2024-Q2.tif,viirs_29000_nara_2024-Q2.tif,1567.878774
2,2,1,1,0.2,0.954862,0.085467,0,3,1,1,...,1,0.290323,2,1,1,0,5,sentinel_2_05000_akita_2020-Q1.tif,viirs_05000_akita_2020-Q1.tif,2009.827701
3,3,4,0,0.2,0.000000,0.119109,1,1,1,0,...,0,0.561290,3,1,2,0,0,sentinel_2_cotabato_2020-Q4.tif,viirs_cotabato_2020-Q4.tif,377.279961
4,4,3,0,0.0,0.002133,0.059541,1,1,1,0,...,1,0.041935,2,1,2,0,1,sentinel_2_pampanga_2019-Q3.tif,viirs_pampanga_2019-Q3.tif,163.905688


In [3]:
class MultiModalDataset(Dataset):
    def __init__(self, df, numeric_cols, categorical_cols,
                 sentinel_col, viirs_col, target_col):
        self.df = df.reset_index(drop=True)
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        self.sentinel_col = sentinel_col
        self.viirs_col = viirs_col
        self.target_col = target_col
        
    def __len__(self):
        return len(self.df)
    
    def load_tiff(self, path):
        img = tiff.imread(path)
        t = torch.tensor(img, dtype=torch.float32)
        if t.dim() == 2:
            # (H, W) → (1, H, W)
            t = t.unsqueeze(0)
        elif t.dim() == 3:
            # (H, W, C) → (C, H, W)
            t = t.permute(2, 0, 1)
        return t
    
    def __getitem__(self, idx):

        x_num = torch.tensor(
            self.df.loc[idx, self.numeric_cols].values.astype("float32"),
            dtype=torch.float32
        )

        x_cat = torch.tensor(
            self.df.loc[idx, self.categorical_cols].values.astype("int64"),
            dtype=torch.long
        )

        img_s = self.load_tiff(ImgPath / self.df.loc[idx, self.sentinel_col])
        img_v = self.load_tiff(ImgPath / self.df.loc[idx, self.viirs_col])

        y = torch.tensor(
            float(self.df.loc[idx, self.target_col]),
            dtype=torch.float32
        )

        return {
            "numeric": x_num,
            "categorical": x_cat,
            "sentinel_img": img_s,
            "viirs_img": img_v,
            "target": y
        }

In [4]:
numeric_cols = [
    'year', 'deflated_gdp_usd', 'us_cpi',
    'straight_distance_to_capital_km'
]

target_col = 'construction_cost_per_m2_usd'

categorical_cols = [
    'access_to_airport', 'access_to_highway', 'access_to_port',
    'access_to_railway', 'country',
    'flood_risk_class', 'geolocation_name', 'koppen_climate_zone',
    'landlocked', 'quarter_label', 'region_economic_classification',
    'seismic_hazard_zone', 'tornadoes_wind_risk',
    'tropical_cyclone_wind_risk'
]

sentinel_col = 'sentinel2_tiff_file_name'
viirs_col = 'viirs_tiff_file_name'

In [5]:
#Convert the columns to the appropriate data types

for col in numeric_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category').cat.codes + 1  # Start categories from 1, reserve 0 for padding

print(train_df[numeric_cols].dtypes)
print(train_df[categorical_cols].dtypes)

year                               float64
deflated_gdp_usd                   float64
us_cpi                             float64
straight_distance_to_capital_km    float64
dtype: object
access_to_airport                 int8
access_to_highway                 int8
access_to_port                    int8
access_to_railway                 int8
country                           int8
flood_risk_class                  int8
geolocation_name                  int8
koppen_climate_zone               int8
landlocked                        int8
quarter_label                     int8
region_economic_classification    int8
seismic_hazard_zone               int8
tornadoes_wind_risk               int8
tropical_cyclone_wind_risk        int8
dtype: object


## Tabular Model (MLP with Embeddings)

In [6]:
class TabularModel(nn.Module):
    def __init__(self, num_numeric, cat_dims, emb_dims):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_dim, emb_dim)
            for cat_dim, emb_dim in zip(cat_dims, emb_dims)
        ])

        emb_total_dim = sum(emb_dims)

        self.mlp = nn.Sequential(
            nn.Linear(num_numeric + emb_total_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

    def forward(self, x_numeric, x_categorical):
        emb = [emb_layer(x_categorical[:, i]) 
               for i, emb_layer in enumerate(self.embeddings)]
        emb = torch.cat(emb, dim=1)

        x = torch.cat([x_numeric, emb], dim=1)
        return self.mlp(x)

## CNN Image Encoder (ResNet Backbone)

In [7]:
class ImageEncoder(nn.Module):
    def __init__(self, in_channels=3, output_dim=128):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, output_dim)

    def forward(self, x):
        return self.backbone(x)

## Fusion Model

In [8]:
class FusionModel(nn.Module):
    def __init__(self, tabular_model, sentinel_model, viirs_model):
        super().__init__()

        self.tabular = tabular_model
        self.sentinel = sentinel_model
        self.viirs = viirs_model

        self.head = nn.Sequential(
            nn.Linear(64 + 128 + 128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x_num, x_cat, img_sentinel, img_viirs):
        t_feat = self.tabular(x_num, x_cat)
        s_feat = self.sentinel(img_sentinel)
        v_feat = self.viirs(img_viirs)

        x = torch.cat([t_feat, s_feat, v_feat], dim=1)
        return self.head(x)

## Loss Function

In [9]:
criterion = nn.SmoothL1Loss()

## Example Model Initialization

In [10]:
num_numeric = len(numeric_cols)
cat_cols = categorical_cols
cat_dims = [train_df[col].nunique() + 1 for col in cat_cols]
emb_dims = [min(50, (dim + 1) // 2) for dim in cat_dims]

tabular_model = TabularModel(num_numeric, cat_dims, emb_dims)
sentinel_model = ImageEncoder(in_channels=12, output_dim=128)
viirs_model = ImageEncoder(in_channels=1,  output_dim=128)

base_model = FusionModel(tabular_model, sentinel_model, viirs_model)

base_model.to(device)

FusionModel(
  (tabular): TabularModel(
    (embeddings): ModuleList(
      (0-5): 6 x Embedding(3, 2)
      (6): Embedding(126, 50)
      (7): Embedding(8, 4)
      (8): Embedding(3, 2)
      (9-10): 2 x Embedding(5, 3)
      (11): Embedding(4, 2)
      (12): Embedding(3, 2)
      (13): Embedding(5, 3)
    )
    (mlp): Sequential(
      (0): Linear(in_features=85, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): ReLU()
    )
  )
  (sentinel): ImageEncoder(
    (backbone): ResNet(
      (conv1): Conv2d(12, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0)

## Training Step (Skeleton)

In [11]:
def train_step(model, optimizer, batch):
    model.train()

    x_num = batch["numeric"].to(device)
    x_cat = batch["categorical"].to(device)
    img_s = batch["sentinel_img"].to(device)
    img_v = batch["viirs_img"].to(device)
    y = batch["target"].to(device)

    preds = model(x_num, x_cat, img_s, img_v)

    loss = criterion(preds.squeeze(), y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()


## Training the model

In [13]:
dataset = MultiModalDataset(train_df, numeric_cols, categorical_cols, sentinel_col, viirs_col, target_col)

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

def train_model(model : torch.nn.Module, loader : DataLoader, epochs=10):
    print("Starting training...")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        total_loss = 0
        batch_count = 0
        for batch in loader:
            print(f"Processing batch {batch_count+1} of {len(loader)}")
            loss = train_step(model, optimizer, batch)
            total_loss += loss
            batch_count += 1
        avg_loss = total_loss / batch_count
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

train_model(base_model, train_loader, epochs=10)

Starting training...
Epoch 1/10
Processing batch 1 of 32


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 25.28 GiB is allocated by PyTorch, and 463.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)